# Sensibilizar el punto exacto: de "estación cercana" a "clima en el punto del cliente" (ECO | Wind)

Hallazgo 24 resolvió cómo encontrar y descargar la estación REAL más cercana a cualquier punto, en
cualquiera de los 20 países del catálogo -- pero "más cercana" no es "exacta" (Bogotá salió a 84.6
km de su estación más cercana). Este notebook prueba el paso que falta: ajustar la MAGNITUD de la
forma real de la estación donante a lo que pasa en el punto EXACTO que pide el cliente, sin volver
a anclar nada a San José ni a ningún sitio fijo.

**Mecanismo propuesto (ver el plan completo en el chat):**

```
media_ajustada_al_punto_exacto = media_real_de_la_estación_donante × [fuente_continua(punto_exacto) / fuente_continua(ubicación_de_la_estación)]
```

La `fuente_continua` sólo se usa para una RAZÓN entre dos puntos cercanos, no para su valor
absoluto -- si esa fuente tiene un sesgo sistemático en la región (NASA POWER subestima ~3x en
Costa Rica, Hallazgo 1), ese sesgo se cancela en gran parte al dividir, y lo que sobrevive es la
diferencia real de microclima entre la estación y el punto exacto. La FORMA horaria (variabilidad,
patrón diurno/estacional) sigue siendo 100% real, de la estación donante -- no se inventa nada,
sólo se reescala la magnitud.

Dos partes: (1) investigar en vivo si el Global Wind Atlas tiene una API de consulta por punto real
y accesible -- no se encontró un endpoint documentado y confirmado en la investigación previa (sólo
"la API existe, no está pensada para bulk"), así que esta parte SÓLO prueba alcanzabilidad de las
páginas reales encontradas, no inventa una URL de datos; (2) el mecanismo completo usando NASA
POWER como fuente continua -- ya confirmado que funciona en Colab (Hallazgo 23), es la vía segura
mientras (1) se termina de confirmar.

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 9e89492 docs(fase2): Hallazgo 24 -- corregir rumbo, la app es internacional (no anclar a San Jose)


/home/user/eco-wind/notebooks
Commit activo: 9e89492  docs(fase2): Hallazgo 24 -- corregir rumbo, la app es internacional (no anclar a San Jose)  (2026-08-31 21:33:44 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import calendar

import numpy as np
import pandas as pd
import requests

from engine.formas_regionales import cargar_formas_conocidas, vecino_mas_cercano
from engine.simulador_pista_a import generar_clima_gwa, simular

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — ¿Tiene el Global Wind Atlas una API de punto real y accesible?

**Honesto de entrada:** la investigación previa (WebSearch) confirmó que GWA 4.0 tiene cobertura
GLOBAL (todos los países + zonas offshore) y que existe un "EMD-API - Global Atlas Services"
documentado como REST/OpenAPI -- pero NO se encontró un endpoint concreto y confirmado para
consulta por punto (lat/lon → media de viento en JSON). `help.emd.dk` (donde vive esa
documentación) ya está confirmado bloqueado en el sandbox de desarrollo (Hallazgo 2) -- acá se
prueba si sigue bloqueado desde Colab, y se revisa el contenido de las páginas reales que sí se
encontraron, para ver si documentan el formato del endpoint. No se inventa ninguna URL de datos.

In [3]:
paginas_reales_a_probar = {
    "Global Wind Atlas (home)": "https://globalwindatlas.info",
    "GWA -- GIS files & API access": "https://globalwindatlas.info/download/gis-files",
    "EMD-API docs (Wiki-WindPRO)": "https://help.emd.dk/mediawiki/index.php/EMD-API_-_Global_Atlas_Services",
    "windatlas.xyz docs (tool de terceros, no es GWA/DTU)": "http://windatlas.xyz/docs/api/",
}

for nombre, url in paginas_reales_a_probar.items():
    try:
        resp = requests.get(url, timeout=15)
        print(f"{nombre}: OK -- {resp.status_code}, {len(resp.text)} caracteres")
        if resp.status_code == 200 and ("api" in resp.text.lower() or "endpoint" in resp.text.lower()):
            print("  (la página menciona 'api'/'endpoint' -- vale la pena leerla completa a mano)")
    except Exception as exc:
        print(f"{nombre}: FALLO -- {exc!r}")

Global Wind Atlas (home): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: / (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


GWA -- GIS files & API access: FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: /download/gis-files (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


EMD-API docs (Wiki-WindPRO): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='help.emd.dk', port=443): Max retries exceeded with url: /mediawiki/index.php/EMD-API_-_Global_Atlas_Services (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
windatlas.xyz docs (tool de terceros, no es GWA/DTU): OK -- 403, 100 caracteres


**Cómo seguir esto si las páginas de arriba SÍ responden en Colab:** leer manualmente la página de
"GIS files & API access" y la de EMD-API para confirmar el formato real de una consulta por punto
(parámetros, autenticación si hace falta, formato de respuesta) antes de escribir código que la
llame -- no se hace acá para no adivinar un contrato de API que después falle en silencio. Si esto
se confirma, GWA sería la mejor opción para la Parte 2 (da una media de viento ya validada por un
modelo físico, sin necesitar corrección estadística). Mientras tanto, la Parte 2 usa NASA POWER,
que ya se sabe que funciona.

## Parte 2 — El mecanismo completo, con NASA POWER como fuente continua (la vía ya confirmada)

`factor_ajuste_nasa_power()`: la razón entre la media de NASA POWER en el punto exacto y en la
ubicación de la estación donante. `evaluar_punto_con_ajuste()`: junta todo -- encuentra el vecino
real más cercano (reusa `vecino_mas_cercano()` de `engine/formas_regionales.py`, Hallazgo 21), y en
vez de escalar su forma a una media "ya conocida" (que en un punto nuevo de verdad NUNCA se tiene),
la escala por este factor de ajuste espacial -- es una prueba más honesta que la de Hallazgo 21/22,
porque no usa ninguna información que no estaría disponible para un punto nuevo real.

In [4]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB", parameters=("WS10M",)):
    params = {
        "parameters": ",".join(parameters), "community": community,
        "longitude": lon, "latitude": lat,
        "start": f"{year}0101", "end": f"{year}1231", "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


def factor_ajuste_nasa_power(lat_exacto, lon_exacto, lat_estacion, lon_estacion, year=2023):
    '''
    Razon NASA POWER(punto exacto) / NASA POWER(ubicacion de la estacion donante) -- el sesgo
    sistematico de NASA POWER (Hallazgo 1) se cancela en gran parte al dividir dos puntos
    cercanos de la misma fuente; sobrevive sobre todo la diferencia real de microclima.
    '''
    media_exacto = fetch_nasa_power_hourly(lat_exacto, lon_exacto, year)["WS10M"].mean()
    media_estacion = fetch_nasa_power_hourly(lat_estacion, lon_estacion, year)["WS10M"].mean()
    return media_exacto / media_estacion, media_exacto, media_estacion


def evaluar_punto_con_ajuste(lat, lon, formas, excluir=None, year=2023,
                              modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_np_exacto, media_np_donante = factor_ajuste_nasa_power(
        lat, lon, donante["lat"], donante["lon"], year=year)

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_np_exacto=media_np_exacto, media_np_donante=media_np_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])

## Validación leave-one-out con el mecanismo nuevo -- ¿mejora sobre lo ya documentado?

Para cada uno de los 4 sitios reales conocidos: se tapa su propia forma Y su propia media real (a
diferencia de Hallazgo 21/22, acá NO se usa la media real ya conocida del sitio -- es información
que un punto nuevo de verdad no tendría). Se compara contra la verdad real ya conocida, y contra
los dos mecanismos ya documentados (siempre San José, y vecino más cercano con curva por residuo
de Hallazgo 22).

In [5]:
formas = cargar_formas_conocidas(usar_residuo=True)
filas = []

for clave, sitio in formas.items():
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                           elevacion_m=sitio["elevacion_m"])
        error_ajustado_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=ajuste["donante"], distancia_km=ajuste["distancia_km"],
                    factor_ajuste_nasa_power=ajuste["factor_ajuste"],
                    kwh_nuevo_ajustado=ajuste["kwh_ajustado"], error_nuevo_ajustado_pct=error_ajustado_pct)
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=None, distancia_km=None, factor_ajuste_nasa_power=None,
                    kwh_nuevo_ajustado=None, error_nuevo_ajustado_pct=f"FALLO: {exc!r}")
    filas.append(fila)

pd.DataFrame(filas)

,sitio,kwh_real,donante,distancia_km,factor_ajuste_nasa_power,kwh_nuevo_ajustado,error_nuevo_ajustado_pct
0,San José (Aeropuerto Juan Santamaría),156.439,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",52.400,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,291.487,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
3,"Finca Favorita (Limón, Caribe)",7.439,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."


## Conclusión

Comparar la columna `error_nuevo_ajustado_pct` de la tabla de arriba contra lo ya documentado:
-41.5%/-43.7%/+19.2% (siempre San José, Hallazgo 21) y +47.6%/+15.9%/+19.2% (vecino más cercano con
curva por residuo, Hallazgo 22) -- esos SÍ usaban la media real ya conocida del sitio (información
que un punto nuevo real no tiene); esta prueba no. Si el mecanismo de ajuste por NASA POWER da un
error parecido o mejor SIN esa ventaja, es una señal fuerte de que generaliza a un punto
verdaderamente nuevo, en cualquiera de los 20 países del catálogo -- no sólo a los 4 sitios donde
ya sabíamos la respuesta. No se declara ganador acá -- correr esto en Colab y ver el número real.